# 🔗 Workflow Integration: End-to-End SAR Processing System

Welcome to the final integration phase! This notebook connects the `RiskAnalystAgent` and `ComplianceOfficerAgent` into a production-ready workflow.

## 🎯 Implementation Strategy
- **Stage 1: Automated Screening**: The `RiskAnalystAgent` processes raw data to flag suspicious patterns.
- **Stage 2: Human-in-the-Loop (HITL)**: A decision gate allows compliance officers to approve or reject cases, optimizing API costs by avoiding narrative generation for non-suspicious cases.
- **Stage 3: Narrative Generation**: The `ComplianceOfficerAgent` uses the **ReACT framework** to draft regulatory-compliant SAR narratives for approved cases only.
- **Stage 4: Audit Logging**: Every transaction, agent decision, and human review is logged to `audit_logs/` for regulatory compliance.

## 🛠️ Instructions to Run
1. Ensure all `src/` modules are implemented and tests are passing.
2. Run the `test_complete_workflow()` function in the final cell.
3. Review the generated output in `/filed_sars/` and `/audit_logs/` folders.

---
*For architectural details, see the project root `README.md`.*

# 🔗 Workflow Integration - Complete SAR Processing System

Welcome to Phase 4 of the Financial Services Agentic AI Project!

In this notebook, you'll integrate both AI agents into a complete **end-to-end SAR processing workflow** that demonstrates real-world financial compliance automation.

## 🎯 Learning Objectives
- Build a complete two-stage AI workflow with human oversight
- Implement human-in-the-loop decision gates for compliance
- Generate complete SAR documents from AI analysis
- Create comprehensive audit trails for regulatory examination
- Demonstrate cost optimization through intelligent agent coordination

## 📋 Business Context
This workflow simulates how banks actually process suspicious activity reports:
1. **Risk Screening**: AI agents analyze transaction patterns for suspicious activity
2. **Human Review**: Compliance officers review AI findings before proceeding
3. **Narrative Generation**: Only approved cases get full compliance documentation
4. **SAR Filing**: Complete regulatory forms are generated for submission
5. **Audit Documentation**: Every decision is logged for regulatory examination

## 🏗️ System Architecture

```
📊 CSV Data → 🔍 Risk Analyst → 👤 Human Decision → ✅ Compliance Officer → 📄 SAR Document
              (Chain-of-Thought)    (Gate)         (ReACT Framework)     (FinCEN Ready)
```

## 🚀 Prerequisites Check

Before starting, ensure you have completed:
- ✅ Phase 1: Foundation components (`foundation_sar.py`)
- ✅ Phase 2: Risk Analyst Agent (`risk_analyst_agent.py`)
- ✅ Phase 3: Compliance Officer Agent (`compliance_officer_agent.py`)
- ✅ Both agents pass their comprehensive test scenarios

If any component is missing, return to previous notebooks to complete implementation.

In [1]:
# Setup and Environment Configuration
import os
import sys
import json
import pandas as pd
import uuid
import hashlib
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../.env')

print("📚 Libraries imported successfully!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path")

📚 Libraries imported successfully!
🔐 Environment variables loaded
📂 Source directory added to Python path


In [2]:
# OpenAI Setup for Vocareum
import openai

# Initialize OpenAI client for Vocareum
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print("📍 Base URL: https://openai.vocareum.com/v1")

✅ OpenAI client initialized with Vocareum routing
🔑 API key: your_ope...here
📍 Base URL: https://openai.vocareum.com/v1


In [3]:
# OpenAI Setup for Vocareum
import openai

# Initialize OpenAI client for Vocareum
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print("📍 Base URL: https://openai.vocareum.com/v1")

✅ OpenAI client initialized with Vocareum routing
🔑 API key: your_ope...here
📍 Base URL: https://openai.vocareum.com/v1


## MOCKING TEST

In [4]:
from mock import mock_agent_output

# Ensure your instance is updated
compliance_agent = mock_agent_output.MockComplianceOfficer()

In [5]:
from IPython.display import clear_output, display
# 1. Import your UI function
from config_manager import get_model_input

clear_output()
# 2. Render the UI
# This will display the dropdown and automatically print selection details 
# whenever you change the selection.
selector, ui_display = get_model_input()
display(ui_display)

Output()

In [6]:

# Check what the user has currently selected in the dropdown
current_choice = selector.value
print(f"The user selected: {current_choice}")


The user selected: reasoning


In [7]:
# TODO: Import Your Implemented Components
# Students: Import your foundation components and agents

print("Uncomment and modify these imports once you've implemented all components:")

from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)
from risk_analyst_agent import RiskAnalystAgent
from compliance_officer_agent import ComplianceOfficerAgent

# TODO: Create agent instances
explainability_logger = ExplainabilityLogger("../outputs/audit_logs/workflow_integration.jsonl")

# ── OPTION A: Mock path (no API funds) ──────────────────────────────
from mock.mock_agent_output import MockRiskAnalyst, MockComplianceOfficer
risk_agent       = MockRiskAnalyst()
compliance_agent = MockComplianceOfficer()
# ── OPTION B: Real API path (uncomment when API is available) ────────
# from config_manager import initialize_agent_system
# from risk_analyst_agent import RiskAnalystAgent
#
# _, _, compliance_agent, model_map = initialize_agent_system(
#     client, explainability_logger, current_choice
# )
# risk_agent = RiskAnalystAgent(
#     client,
#     explainability_logger,
#     model=model_map['expert']
# )
# print(f"✅ risk_agent       → {model_map['expert']}")
# print(f"✅ compliance_agent → {model_map['compliance']}")


print("✅ Ready to import components after implementation")

Uncomment and modify these imports once you've implemented all components:
✅ Ready to import components after implementation


## 📊 Step 1: Data Loading and Preprocessing

Load the financial data and prepare it for analysis.

In [8]:
# Stage 1: Load
def load_csv_files(data_dir="../data/"):
    print("📂 Stage 1: Loading CSV files...")
    # Using consistent variable names: df_customers, df_accounts, df_transactions
    df_customers = pd.read_csv(f"{data_dir}/customers.csv", dtype={'ssn_last_4': str})
    df_accounts = pd.read_csv(f"{data_dir}/accounts.csv")
    df_transactions = pd.read_csv(f"{data_dir}/transactions.csv")
    print("✅ Files loaded successfully.")
    return df_customers, df_accounts, df_transactions

# Stage 2: Clean
def clean_data(df_customers, df_accounts, df_transactions):
    print("🧹 Stage 2: Cleaning missing values...")
    # Consistent usage of the 'df' list for the loop
    for df in [df_customers, df_accounts, df_transactions]:
        df.fillna('', inplace=True)
    print("✅ Data cleaned.")
    return df_customers, df_accounts, df_transactions

# Stage 3: Convert
def convert_to_dicts(df_customers, df_accounts, df_transactions):
    print("🔄 Stage 3: Converting to dictionary format...")
    # Consistent output naming: data_customers, data_accounts, data_transactions
    data_customers = df_customers.to_dict('records')
    data_accounts = df_accounts.to_dict('records')
    data_transactions = df_transactions.to_dict('records')
    print("✅ Conversion complete.")
    return data_customers, data_accounts, data_transactions

In [9]:
# TODO: Load and Preprocess Financial Data
# Students: Load customer, account, and transaction data

def load_and_preprocess_data():
    """
    Load CSV data and prepare for analysis
    
    This function should:
    1. Load customers.csv, accounts.csv, transactions.csv
    2. Handle missing values appropriately
    3. Create data dictionaries for processing
    4. Return cleaned datasets
    """
    print("📊 Loading Financial Data")
    df_cust, df_acc, df_trans = load_csv_files()    
    # Handle NaN values
    df_cust, df_acc, df_trans = clean_data(df_cust, df_acc, df_trans)    
    # Convert to dictionaries
    data_cust, data_acc, data_trans = convert_to_dicts(df_cust, df_acc, df_trans)    

    
    print(f"📈 Loaded: {len(data_cust)} customers, {len(data_acc)} accounts, {len(data_trans)} transactions")    
    return data_cust, data_acc, data_trans    
    # return None, None, None

# Load data
customers_data, accounts_data, transactions_data = load_and_preprocess_data()

📊 Loading Financial Data
📂 Stage 1: Loading CSV files...
✅ Files loaded successfully.
🧹 Stage 2: Cleaning missing values...
✅ Data cleaned.
🔄 Stage 3: Converting to dictionary format...
✅ Conversion complete.
📈 Loaded: 150 customers, 178 accounts, 4268 transactions


## 🎯 Step 2: Customer Risk Screening

Implement intelligent customer screening to identify high-risk cases for detailed analysis.

In [10]:
def screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=5):
    """
    TODO: Implement risk-based customer screening
    
    Screening criteria should include:
    1. High risk ratings (Medium, High)
    2. Large transaction amounts (>$100K total)
    3. High transaction frequency (>50 transactions)
    4. Recent activity patterns
    
    Returns top N highest-risk customers for detailed analysis
    """
    print("🔍 Customer Risk Screening")

    print(f"🔍 Screening {len(customers_data)} customers for high-risk flags...")
    # Example screening logic (uncomment and modify):
    selected_customers = []
    
    for customer in customers_data:
        # Get customer accounts and transactions
        customer_accounts = [acc for acc in accounts_data if acc['customer_id'] == customer['customer_id']]

        # Filter transactions based on the accounts linked to this customer
        account_ids = [acc['account_id'] for acc in customer_accounts]
        customer_transactions = [txn for txn in transactions_data if txn['account_id'] in account_ids]   
            
        # Calculate risk indicators
        total_amount = sum(abs(txn['amount']) for txn in customer_transactions if txn['amount'] != '')
        transaction_count = len(customer_transactions)
        risk_rating = customer['risk_rating']
        
        # Apply screening criteria
        risk_flags = []
        if risk_rating in ['Medium', 'High']:
            risk_flags.append('high_risk_rating')
        if total_amount > 100000:
            risk_flags.append('large_amounts')
        if transaction_count > 50:
            risk_flags.append('high_frequency')
        
        # Select high-risk customers
        if len(risk_flags) >= 2:  # Multiple risk flags
            selected_customers.append({
                'customer': customer,
                'accounts': customer_accounts,
                'transactions': customer_transactions,
                'total_amount': total_amount,
                'transaction_count': transaction_count,
                'risk_flags': risk_flags
            })
    
    # Sort by risk score and take top N
    selected_customers.sort(key=lambda x: (len(x['risk_flags']), x['total_amount']), reverse=True)
    return selected_customers[:top_n]
    
    print(f"📊 Selected 0 customers for analysis (implement screening logic)")
    return []
# Run customer screening
selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data)

🔍 Customer Risk Screening
🔍 Screening 150 customers for high-risk flags...


In [11]:

# Pass the data you just loaded into your screening function
selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data)

# Print how many were found
print(f"Number of high-risk customers identified: {len(selected_customers)}")

🔍 Customer Risk Screening
🔍 Screening 150 customers for high-risk flags...
Number of high-risk customers identified: 5


## 📄 Step 3: SAR Document Generation

Create complete, FinCEN-ready SAR documents with all required metadata.

In [12]:
def create_sar_document(case_data, risk_analysis, compliance_review):
        """
        TODO: Create complete SAR document
        
        SAR document should include:
        1. SAR metadata (ID, filing date, type, checksum)
        2. Subject information (customer details)
        3. Suspicious activity description
        4. AI analysis results
        5. Compliance narrative
        6. Regulatory citations
        7. Filing institution information
        """
        print("📄 Creating SAR Document")
        
        # Example SAR document structure (uncomment and implement):
        sar_id = f"SAR_{uuid.uuid4().hex[:8]}"
        filing_date = datetime.now().isoformat()
        
        sar_document = {
            'sar_metadata': {
                'sar_id': sar_id,
                'filing_date': filing_date,
                'filing_type': 'Suspicious Activity Report',
                'ai_generated': True,
                'review_status': 'human_approved'
            },
            'subject_information': {
                'customer_name': case_data.customer.name,
                'customer_id': case_data.customer.customer_id,
                'address': case_data.customer.address,
                'customer_since': case_data.customer.customer_since,
                'risk_rating': case_data.customer.risk_rating
            },
            'suspicious_activity': {
                'classification': risk_analysis.classification,
                'risk_level': risk_analysis.risk_level,
                'confidence_score': risk_analysis.confidence_score,
                'narrative': compliance_review.narrative,
                'key_indicators': risk_analysis.key_indicators,
                'ai_reasoning': risk_analysis.reasoning
            },
            'regulatory_compliance': {
                'citations': getattr(compliance_review, 'regulatory_citations', []),
                'narrative_word_count': len(compliance_review.narrative.split()),
                'compliance_status': 'approved'
            },
            'audit_trail': {
                'case_id': case_data.case_id,
                'processing_date': filing_date,
                'ai_agents_used': ['RiskAnalyst', 'ComplianceOfficer'],
                'human_reviewer': 'compliance_officer'
            }
        }
        
        return sar_document
        
        # return {}

def save_sar_document(sar_document):
    """TODO: Save SAR document to outputs directory"""
    # 1. Define the absolute path for the target directory
    # Using 'abspath' ensures we always point to the same location, 
    # regardless of where the notebook is running from.
    output_dir = "../outputs/filed_sars"
    os.makedirs(output_dir, exist_ok=True)
    
    filename = f"{output_dir}/{sar_document['sar_metadata']['sar_id']}.json"
    
    try:
        with open(filename, 'w') as f:
            json.dump(sar_document, f, indent=4)
        print(f"✅ SAR successfully saved: {filename}")
    except IOError as e:
        print(f"❌ Failed to save SAR document: {e}")
        
print("📄 SAR document generation functions defined")

📄 SAR document generation functions defined


In [13]:
    # 1. HELPER: Isolated input logic
    def get_human_decision(risk_level):
        """Forces input and provides a fallback to prevent infinite loops."""
        # Add a print to ensure you see the prompt
        print("\n--- 🛑 ACTION REQUIRED ---")
        
        # We use a simple count to break the loop if it gets stuck
        attempts = 0
        while attempts < 3:
            decision = input(f"\n\t| Risk_Level Associated with this Case: {risk_level} | \n\t🤔 Proceed with SAR filing? (yes/no): |").strip().lower()
            if decision in ['yes', 'y', '']: 
                return True
            if decision in ['no', 'n']: 
                return False
            
            print(f"Invalid input. Attempt {attempts+1}/3. Please enter 'yes' or 'no'.")
            attempts += 1
        
        # If the user is having trouble, default to 'no' to keep the pipeline moving
        print("⚠️ Input failed, defaulting to 'no'.")
        return False

In [14]:
def process_single_customer(i, total_customers, customer_data, risk_agent, compliance_agent, approved_sars, rejected_cases, audit_decisions):
    """Encapsulates the two-stage analysis and filing logic."""
    
    # 1. DEFENSIVE TYPE-CHECKING: Handle mixed input formats (dict vs Object)
    if isinstance(customer_data, dict):
        loader = DataLoader(explainability_logger)
        case_data = loader.create_case_from_data(
            customer_data['customer'],
            customer_data['accounts'], 
            customer_data['transactions']
        )
    else:
        # It is already a CaseData object
        case_data = customer_data
        
    print(f"DEBUG: Type of case_data.customer is {type(case_data.customer)}")
    
    # 2. Stage 1: Risk Analysis
    risk_analysis = risk_agent.analyze_case(case_data)
    print(f"Risk Level: {risk_analysis.risk_level}")


    # 2. Minimalist UI: Display ONLY what you need to see when asking for input
    # Access the attributes directly from your existing risk_analysis object
    print(f"\n--- [Case {i}/{total_customers}] ---")
    print(f"Risk Level: {risk_analysis.risk_level}")
    # Note: If your narrative is stored in a different attribute (like .reasoning or .analysis),
    # just change .narrative to that attribute name.
    # print(f"Narrative: {risk_analysis.narrative}")    
    # 3. Decision Gate (Hard-coded to True to bypass input issues)
    # should_proceed = True
    should_proceed = get_human_decision(risk_analysis.risk_level) # <--- HERE!

    compliance_review = None
    sar_doc = None
    
    # 4. Stage 2: Compliance and Filing
    if should_proceed:
        # Generate the review FIRST
        compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
        # Pass the generated review to the document creator
        sar_doc = create_sar_document(case_data, risk_analysis, compliance_review)        
        save_sar_document(sar_doc)
        approved_sars.append(case_data.case_id)
    else:
        rejected_cases.append(case_data.case_id)
        
    # 5. Audit Logging
    audit_decisions.append({
        'case_id': case_data.case_id,
        'customer_name': case_data.customer.name,
        'decision': 'PROCEED' if should_proceed else 'REJECT',
        'ai_classification': risk_analysis.classification,
        'ai_confidence': risk_analysis.confidence_score,
        'compliance_narrative_exists': compliance_review is not None 
    })
        
    # Return all required variables to the main workflow
    return should_proceed, case_data, risk_analysis, sar_doc

## 🤖 Step 4: Two-Stage AI Analysis with Human Gates

Implement the core two-stage workflow:
1. **Stage 1**: Risk Analyst performs Chain-of-Thought analysis
2. **Human Gate**: Review and decision to proceed
3. **Stage 2**: Compliance Officer generates ReACT narratives (only if approved)

In [15]:
def run_two_stage_sar_workflow(selected_customers):
    """
    TODO: Implement complete two-stage SAR processing workflow
    
    For each customer:
    1. Create CaseData object
    2. Run Risk Analyst analysis (Chain-of-Thought)
    3. Present findings to human reviewer
    4. Get human decision (proceed/reject)
    5. If approved: Run Compliance Officer (ReACT)
    6. Generate complete SAR document
    7. Log all decisions for audit
    """
    print("🤖 Two-Stage SAR Processing Workflow")
    
    # Initialize tracking
    processed_cases, approved_sars, rejected_cases, audit_decisions = [], [], [], []
    
    print("   1. Create CaseData from customer, accounts, transactions")
    print("   2. Run Risk Analyst analysis")
    print("   3. Display analysis results to human reviewer")
    print("   4. Get human decision (input('Proceed with SAR filing? (yes/no): '))")
    print("   5. If 'yes': Run Compliance Officer narrative generation")
    print("   6. Create complete SAR document with all metadata")
    print("   7. Save SAR to ../outputs/filed_sars/ directory")
    print("   8. Log decision to audit trail")
    
    # Example workflow structure (uncomment and implement):
    for i, customer_data in enumerate(selected_customers, 1):
        try:
            # Place prints INSIDE the try block so they are part of the logged attempt
            print(f"\n--- Processing index {i} ---")
            print(f"DEBUG: Input type is {type(customer_data)}")
            should_proceed, case_data, risk_analysis, sar_doc = process_single_customer(
                i, len(selected_customers), customer_data, 
                risk_agent, compliance_agent, 
                approved_sars, rejected_cases, audit_decisions
            )
            if should_proceed:
                processed_cases.append(case_data.customer.customer_id)        
        except Exception as e:
            print(f"❌ Error processing customer: {e}")
    
    return processed_cases, approved_sars, rejected_cases, audit_decisions
    
processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)


🤖 Two-Stage SAR Processing Workflow
   1. Create CaseData from customer, accounts, transactions
   2. Run Risk Analyst analysis
   3. Display analysis results to human reviewer
   4. Get human decision (input('Proceed with SAR filing? (yes/no): '))
   5. If 'yes': Run Compliance Officer narrative generation
   6. Create complete SAR document with all metadata
   7. Save SAR to ../outputs/filed_sars/ directory
   8. Log decision to audit trail

--- Processing index 1 ---
DEBUG: Input type is <class 'dict'>
DEBUG: Type of case_data.customer is <class 'foundation_sar.CustomerData'>
Risk Level: High

--- [Case 1/5] ---
Risk Level: High

--- 🛑 ACTION REQUIRED ---
📄 Creating SAR Document
✅ SAR successfully saved: ../outputs/filed_sars/SAR_907a7dd0.json

--- Processing index 2 ---
DEBUG: Input type is <class 'dict'>
DEBUG: Type of case_data.customer is <class 'foundation_sar.CustomerData'>
Risk Level: High

--- [Case 2/5] ---
Risk Level: High

--- 🛑 ACTION REQUIRED ---
📄 Creating SAR Document

## 📊 Step 5: Workflow Metrics and Analysis

Analyze the efficiency and effectiveness of your AI-powered SAR processing system.

In [16]:
def analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions):
        """
        TODO: Calculate workflow efficiency metrics
        
        Metrics to calculate:
        1. Processing efficiency (time per case)
        2. Cost optimization (two-stage vs single-stage)
        3. Human decision patterns
        4. AI accuracy validation
        5. Regulatory compliance rates
        """
        print("📊 Workflow Efficiency Analysis")
        print("📋 Calculate processing metrics")
        
        # Example metrics calculation (uncomment and implement):
        total_cases = len(processed_cases)
        approved_cases = len(approved_sars)
        rejected_cases_count = len(rejected_cases)
        
        if total_cases > 0:
            approval_rate = approved_cases / total_cases
            rejection_rate = rejected_cases_count / total_cases
        else:
            approval_rate = rejection_rate = 0
        
        print(f"📈 WORKFLOW METRICS:")
        print(f"   Total Cases Processed: {total_cases}")
        print(f"   SARs Filed: {approved_cases}")
        print(f"   Cases Rejected: {rejected_cases_count}")
        print(f"   Approval Rate: {approval_rate:.1%}")
        print(f"   Rejection Rate: {rejection_rate:.1%}")
        


        # Cost savings Calculation
        # Assume Risk Analyst = $0.50, Compliance Officer = $5.00
        risk_cost = total_cases * 0.50
        compliance_cost = approved_cases * 5.00
        total_spend = risk_cost + compliance_cost    
        # Cost optimization analysis
        print(f"\n💰 COST OPTIMIZATION:")
        print(f"   Two-stage processing saves costs by only running")
        print(f"   expensive compliance generation on approved cases")
        print(f"   Cost savings: {rejection_rate:.1%} of compliance calls avoided")
        
def validate_ai_decisions(audit_decisions):
    """TODO: Analyze AI decision patterns and accuracy"""
    if not audit_decisions:
        print("   No audit decisions available to validate.")
        return

    # Extracting data from your audit structure
    confidences = [d['ai_confidence'] for d in audit_decisions]
    avg_confidence = sum(confidences) / len(confidences)
    
    print(f"📋 Analyze confidence score distributions:")
    print(f"   Average AI Model Confidence: {avg_confidence:.2%}")
    
    print("📋 Review human override patterns:")
    # Logic to compare decision vs classification
    # Count interactions
    human_interactions = [d for d in audit_decisions if d['decision'] in ['PROCEED', 'REJECT']]
    manual_rejections = [d for d in audit_decisions if d['decision'] == 'REJECT']
    
    print(f"\n📋 Validate AI classification accuracy")
    print(f"   Total Human Decisions Made: {len(human_interactions)}")
    
    print(f"📋 Review human override patterns:")
    print(f"   Manual Rejections (Human Gate): {len(manual_rejections)} cases.")
    
    for d in manual_rejections:
        # Displaying both Case ID and Customer ID
        print(f"   - Case ID: {d['case_id']}")
# Run analysis
analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
validate_ai_decisions(audit_decisions)

📊 Workflow Efficiency Analysis
📋 Calculate processing metrics
📈 WORKFLOW METRICS:
   Total Cases Processed: 5
   SARs Filed: 5
   Cases Rejected: 0
   Approval Rate: 100.0%
   Rejection Rate: 0.0%

💰 COST OPTIMIZATION:
   Two-stage processing saves costs by only running
   expensive compliance generation on approved cases
   Cost savings: 0.0% of compliance calls avoided
📋 Analyze confidence score distributions:
   Average AI Model Confidence: 85.00%
📋 Review human override patterns:

📋 Validate AI classification accuracy
   Total Human Decisions Made: 5
📋 Review human override patterns:
   Manual Rejections (Human Gate): 0 cases.


## 🏁 Step 6: Complete System Demonstration

Test your complete system with comprehensive scenarios to validate production readiness.

In [17]:
!PYTHONPATH=. pytest ../tests/test_compliance_officer.py -v -s

============================= test session starts ==============================
platform linux -- Python 3.13.0, pytest-8.4.1, pluggy-1.6.0 -- /opt/venv/bin/python
cachedir: .pytest_cache
rootdir: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter
plugins: anyio-4.10.0
collecting ... DEBUG [ComplianceOfficerAgent]: Received client ID 136433597346016
collected 10 items                                                             

../tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_agent_initialization DEBUG [ComplianceOfficerAgent]: Received client ID 136434479255232
PASSED
../tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success DEBUG [ComplianceOfficerAgent]: Received client ID 136434479255232
PASSED
../tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_narrative_word_count_validation DEBUG [ComplianceOfficerAgent]: Received client ID 136433527885168
PASSED
../tests/test_compliance_offi

In [18]:
# TODO: Run Complete System Test
# Students: Demonstrate your complete SAR processing system

def demonstrate_complete_system():
    """
    TODO: Run complete system demonstration
    
    This should:
    1. Process multiple customers through the complete workflow
    2. Show both approved and rejected cases
    3. Generate multiple SAR documents
    4. Demonstrate audit trail creation
    5. Show efficiency metrics
    """
    print("🏁 Complete SAR Processing System Demonstration")
    
    # Example demonstration (uncomment after implementation):
    print("🚀 Running complete system test...")
    
    # Load fresh data
    customers_data, accounts_data, transactions_data = load_and_preprocess_data()
    
    # Screen customers
    selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=3)
    
    # Run workflow
    processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)
    
    # Generate final report
    analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
    
    print(f"🎉 System demonstration complete!")
    print(f"📄 SAR documents saved to: ../outputs/filed_sars/")
    print(f"📊 Audit logs saved to: ../outputs/audit_logs/")
    
demonstrate_complete_system()

🏁 Complete SAR Processing System Demonstration
🚀 Running complete system test...
📊 Loading Financial Data
📂 Stage 1: Loading CSV files...
✅ Files loaded successfully.
🧹 Stage 2: Cleaning missing values...
✅ Data cleaned.
🔄 Stage 3: Converting to dictionary format...
✅ Conversion complete.
📈 Loaded: 150 customers, 178 accounts, 4268 transactions
🔍 Customer Risk Screening
🔍 Screening 150 customers for high-risk flags...
🤖 Two-Stage SAR Processing Workflow
   1. Create CaseData from customer, accounts, transactions
   2. Run Risk Analyst analysis
   3. Display analysis results to human reviewer
   4. Get human decision (input('Proceed with SAR filing? (yes/no): '))
   5. If 'yes': Run Compliance Officer narrative generation
   6. Create complete SAR document with all metadata
   7. Save SAR to ../outputs/filed_sars/ directory
   8. Log decision to audit trail

--- Processing index 1 ---
DEBUG: Input type is <class 'dict'>
DEBUG: Type of case_data.customer is <class 'foundation_sar.Custome

## 📝 Implementation Checklist

### ✅ Workflow Integration Deliverables
- [ ] **Data Loading**: Load and preprocess CSV data with proper error handling
- [ ] **Customer Screening**: Implement risk-based screening to identify high-risk cases
- [ ] **Two-Stage Workflow**: Build complete Risk Analyst → Human Gate → Compliance Officer flow
- [ ] **Human Decision Gates**: Implement interactive approval/rejection points
- [ ] **SAR Document Generation**: Create complete FinCEN-ready documents with metadata
- [ ] **Audit Trail Creation**: Log all decisions and reasoning for regulatory examination
- [ ] **Efficiency Metrics**: Calculate cost optimization and processing efficiency
- [ ] **System Demonstration**: Test complete workflow with multiple scenarios

### ✅ Testing and Validation Requirements
- [ ] **Component Validation**: Verify all foundation components and agents are available
- [ ] **Integration Testing**: Run comprehensive test suites for all components with proper sys.path setup
- [ ] **End-to-End Testing**: Test complete workflow with automated scenarios
- [ ] **Error Handling Testing**: Validate graceful handling of edge cases and failures
- [ ] **Output Validation**: Ensure SAR documents meet regulatory standards
- [ ] **Performance Testing**: Measure workflow efficiency and processing times

### ✅ Technical Requirements
- [ ] **Error Handling**: Robust exception handling for all workflow steps
- [ ] **Data Validation**: Proper validation of all inputs and outputs
- [ ] **File Management**: Organize outputs in appropriate directories
- [ ] **Logging**: Comprehensive audit logging for compliance
- [ ] **Performance**: Efficient processing of multiple cases
- [ ] **User Experience**: Clear prompts and feedback for human reviewers
- [ ] **Test Infrastructure**: Proper test imports and sys.path configuration

### ✅ Business Requirements  
- [ ] **Regulatory Compliance**: Ensure all SAR documents meet FinCEN requirements
- [ ] **Cost Optimization**: Demonstrate savings from two-stage processing
- [ ] **Audit Readiness**: Create examination-ready documentation
- [ ] **Quality Assurance**: Validate AI decisions with human oversight
- [ ] **Scalability**: Design for processing larger datasets
- [ ] **Production Readiness**: Complete testing validates system reliability

## 🎯 Success Criteria

By completion, your integrated system should:
- ✅ Process real financial data with proper validation
- ✅ Execute complete two-stage AI workflow with human gates
- ✅ Generate regulatory-compliant SAR documents
- ✅ Create comprehensive audit trails for all decisions
- ✅ Demonstrate measurable cost optimization benefits
- ✅ Handle errors gracefully and provide clear user feedback
- ✅ Pass all integration and end-to-end tests
- ✅ Meet production-ready quality standards

## 🚀 Next Steps

1. **Complete Implementation**: Fill in all TODO sections with working code
2. **Run Integration Tests**: Validate all components work together properly
3. **Execute End-to-End Tests**: Test complete workflow with automated scenarios
4. **Test Thoroughly**: Run complete workflow with various manual scenarios
5. **Validate Outputs**: Ensure SAR documents meet regulatory requirements
6. **Document Results**: Create final project documentation and metrics
7. **Prepare Presentation**: Demonstrate your system's capabilities and business value

**Congratulations on building a complete AI-powered SAR processing system! 🎉**

## 🧪 Step 7: Workflow Testing and Validation

Before finalizing your implementation, validate your complete system with comprehensive testing.

In [19]:
# 🧪 Workflow Integration Testing
# Validate your complete system with integration tests

import sys
import os

# Add tests directory to Python path for importing test modules
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Added tests directory to Python path: {tests_path}")

def run_integration_tests():
    """
    Run comprehensive integration tests to validate the complete workflow
    
    Tests include:
    1. Foundation components integration
    2. Agent communication and data flow
    3. End-to-end workflow execution
    4. Error handling and edge cases
    5. Output validation and compliance
    """
    print("🧪 Comprehensive Integration Testing")
    
    # Uncomment when your complete system is ready:
    try:
        # Import all test modules
        from test_foundation import TestCustomerData, TestAccountData, TestTransactionData, TestCaseData
        from test_risk_analyst import TestRiskAnalystAgent
        from test_compliance_officer import TestComplianceOfficerAgent
        import pytest
        
        print("🔍 Running Foundation Component Tests...")
        foundation_result = pytest.main([
            f"{tests_path}/test_foundation.py", 
            "-v", 
            "--tb=short"
        ])
        
        print("🔍 Running Risk Analyst Agent Tests...")
        risk_result = pytest.main([
            f"{tests_path}/test_risk_analyst.py", 
            "-v", 
            "--tb=short"
        ])
        
        print("📝 Running Compliance Officer Agent Tests...")
        compliance_result = pytest.main([
            f"{tests_path}/test_compliance_officer.py", 
            "-v", 
            "--tb=short"
        ])
        
        # Calculate overall test results
        all_passed = foundation_result == 0 and risk_result == 0 and compliance_result == 0
        
        print("\n" + "="*60)
        print("📊 INTEGRATION TEST RESULTS:")
        print(f"   Foundation Components: {'✅ PASS' if foundation_result == 0 else '❌ FAIL'}")
        print(f"   Risk Analyst Agent: {'✅ PASS' if risk_result == 0 else '❌ FAIL'}")
        print(f"   Compliance Officer Agent: {'✅ PASS' if compliance_result == 0 else '❌ FAIL'}")
        print(f"   Overall Status: {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'}")
        
        if all_passed:
            print("\n🎉 Your system is ready for production workflow testing!")
            print("📝 Proceed to run the complete system demonstration.")
        else:
            print("\n⚠️ Fix failing tests before running the complete workflow.")
            print("📝 Return to previous notebooks to fix component issues.")
        
        return all_passed
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure all components are implemented:")
        print("   • foundation_sar.py")
        print("   • risk_analyst_agent.py") 
        print("   • compliance_officer_agent.py")
        return False

def validate_workflow_components():
    """Validate that all required components are available for integration"""
    print("🔍 Validating Workflow Components")
    
    components_status = {
        'foundation_sar': False,
        'risk_analyst_agent': False,
        'compliance_officer_agent': False,
        'test_modules': False
    }
    
    try:
        # Check foundation components
        from foundation_sar import CustomerData, CaseData, ExplainabilityLogger, DataLoader
        components_status['foundation_sar'] = True
        print("✅ Foundation components available")
    except ImportError:
        print("❌ Foundation components not available")
    
    try:
        # Check risk analyst agent
        from risk_analyst_agent import RiskAnalystAgent
        components_status['risk_analyst_agent'] = True
        print("✅ Risk Analyst Agent available")
    except ImportError:
        print("❌ Risk Analyst Agent not available")
    
    try:
        # Check compliance officer agent
        from compliance_officer_agent import ComplianceOfficerAgent
        components_status['compliance_officer_agent'] = True
        print("✅ Compliance Officer Agent available")
    except ImportError:
        print("❌ Compliance Officer Agent not available")
    
    try:
        # Check test modules
        from test_foundation import TestCustomerData
        from test_risk_analyst import TestRiskAnalystAgent  
        from test_compliance_officer import TestComplianceOfficerAgent
        components_status['test_modules'] = True
        print("✅ Test modules available")
    except ImportError:
        print("❌ Test modules not available")
    
    all_ready = all(components_status.values())
    
    print(f"\n📊 Component Status: {'✅ ALL READY' if all_ready else '⚠️ INCOMPLETE'}")
    if not all_ready:
        print("💡 Complete missing components before running integration tests")
    
    return all_ready

# Run component validation
components_ready = validate_workflow_components()

# Run integration tests if components are ready
if components_ready:
    print("\n🚀 All components ready - you can run integration tests!")
    run_integration_tests()
else:
    print("\n📋 Complete component implementation first, then run integration tests")

📁 Added tests directory to Python path: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter/tests
🔍 Validating Workflow Components
✅ Foundation components available
✅ Risk Analyst Agent available
✅ Compliance Officer Agent available
DEBUG [RiskAnalystAgent]: Received client ID 139513897853888
DEBUG [ComplianceOfficerAgent]: Received client ID 139513897857248
✅ Test modules available

📊 Component Status: ✅ ALL READY

🚀 All components ready - you can run integration tests!
🧪 Comprehensive Integration Testing
🔍 Running Foundation Component Tests...
============================= test session starts ==============================
platform linux -- Python 3.13.0, pytest-8.4.1, pluggy-1.6.0 -- /opt/venv/bin/python
cachedir: .pytest_cache
rootdir: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter
plugins: anyio-4.10.0
collecting ... collected 10 items

../tests/test_foundation.py::TestCustomerData::test_valid_customer_data PASSED [ 10%]
../tests/test_foundation.py::

In [20]:
# 🎯 End-to-End Workflow Testing
# Test the complete workflow with known test scenarios

def test_complete_workflow():
    """
    Test the complete SAR processing workflow end-to-end
    
    This test should:
    1. Load test data (can use a subset of actual data)
    2. Run customer screening
    3. Execute two-stage AI analysis
    4. Simulate human decisions (automated for testing)
    5. Generate SAR documents
    6. Validate all outputs
    7. Check audit trail completeness
    """
    print("🎯 End-to-End Workflow Testing")
    
    # Example end-to-end test structure (uncomment when ready):
    try:
        print("🚀 Starting end-to-end workflow test...")
        
        # Test data preparation
        print("📊 Loading test data...")
        customers_data, accounts_data, transactions_data = load_and_preprocess_data()
        
        if not customers_data:
            print("⚠️ No test data available - implement data loading first")
            return False
        
        # Test customer screening
        print("🔍 Testing customer screening...")
        selected_customers = screen_high_risk_customers(
            customers_data, accounts_data, transactions_data, top_n=2
        )
        
        if not selected_customers:
            print("⚠️ No customers selected - check screening criteria")
            return False
        
        print(f"✅ Selected {len(selected_customers)} customers for testing")
        
        # Test workflow with automated decisions (for testing)
        print("🤖 Testing automated workflow...")
        test_results = {
            'cases_processed': 0,
            'sars_generated': 0,
            'errors': []
        }
        
        for customer_data in selected_customers:
            try:
                # Create case
                loader = DataLoader(explainability_logger)
                case_data = loader.create_case_from_data(
                    customer_data['customer'],
                    customer_data['accounts'],
                    customer_data['transactions']
                )
                
                # Test Risk Analyst
                risk_analysis = risk_agent.analyze_case(case_data)
                assert hasattr(risk_analysis, 'classification'), "Risk analysis missing classification"
                assert hasattr(risk_analysis, 'confidence_score'), "Risk analysis missing confidence"
                
                # Test Compliance Officer (simulate approval)
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                assert hasattr(compliance_review, 'narrative'), "Compliance review missing narrative"
                
                # Test SAR generation
                sar_document = create_sar_document(case_data, risk_analysis, compliance_review)
                save_sar_document(sar_document) #added extra code to sar is saved 
                assert sar_document, "SAR document generation failed"
                
                test_results['cases_processed'] += 1
                test_results['sars_generated'] += 1
                
                print(f"✅ Successfully processed: {customer_data['customer']['name']}")
                
            except Exception as e:
                test_results['errors'].append(f"Error processing {customer_data['customer']['name']}: {e}")
                print(f"❌ Error processing: {customer_data['customer']['name']}: {e}")
        
        # Test results summary
        print("\n📊 END-TO-END TEST RESULTS:")
        print(f"   Cases Processed: {test_results['cases_processed']}")
        print(f"   SARs Generated: {test_results['sars_generated']}")
        print(f"   Errors: {len(test_results['errors'])}")
        
        if test_results['errors']:
            print("❌ Test Errors:")
            for error in test_results['errors']:
                print(f"     • {error}")
        
        success = len(test_results['errors']) == 0 and test_results['cases_processed'] > 0
        
        if success:
            print("\n🎉 END-TO-END TEST PASSED!")
            print("✅ Your complete workflow is ready for production use!")
        else:
            print("\n⚠️ END-TO-END TEST HAD ISSUES")
            print("📝 Fix the errors above before deploying to production")
        
        return success
        
    except Exception as e:
        print(f"❌ End-to-end test failed: {e}")
        print("💡 Ensure all components are properly implemented")
        return False

# Run end-to-end test
print("📋 Running end-to-end test after implementing complete workflow")
test_success = test_complete_workflow()

📋 Running end-to-end test after implementing complete workflow
🎯 End-to-End Workflow Testing
🚀 Starting end-to-end workflow test...
📊 Loading test data...
📊 Loading Financial Data
📂 Stage 1: Loading CSV files...
✅ Files loaded successfully.
🧹 Stage 2: Cleaning missing values...
✅ Data cleaned.
🔄 Stage 3: Converting to dictionary format...
✅ Conversion complete.
📈 Loaded: 150 customers, 178 accounts, 4268 transactions
🔍 Testing customer screening...
🔍 Customer Risk Screening
🔍 Screening 150 customers for high-risk flags...
✅ Selected 2 customers for testing
🤖 Testing automated workflow...
📄 Creating SAR Document
✅ SAR successfully saved: ../outputs/filed_sars/SAR_65df5c1e.json
✅ Successfully processed: Jacqueline Rodriguez
📄 Creating SAR Document
✅ SAR successfully saved: ../outputs/filed_sars/SAR_c4900cfa.json
✅ Successfully processed: Michael Stanley

📊 END-TO-END TEST RESULTS:
   Cases Processed: 2
   SARs Generated: 2
   Errors: 0

🎉 END-TO-END TEST PASSED!
✅ Your complete workflow 

In [21]:
# Integration smoke test for DataLoader → RiskAnalystAgent → ComplianceOfficerAgent
# CHANGED: Verifies that risk_analysis and case_data flow correctly into Compliance Officer.

import json
import math


from src.foundation_sar import load_csv_data, DataLoader, ExplainabilityLogger
from src.risk_analyst_agent import RiskAnalystAgent
from src.compliance_officer_agent import ComplianceOfficerAgent

print("Running end-to-end integration smoke test...\n")

# ---------- 1) Load one sample case via DataLoader ----------
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir = os.path.join(project_root, "data")
print("Using data_dir:", data_dir)

customers_df, accounts_df, transactions_df = load_csv_data(data_dir=data_dir)

# ---------- 1) Load one sample case via DataLoader ----------

customers_df, accounts_df, transactions_df = load_csv_data(data_dir=data_dir)

# CHANGED: helper to find a customer with at least one transaction
def get_case_data_with_transactions():
    for _, cust_row in customers_df.iterrows():
        cust = cust_row.to_dict()
        cid = cust["customer_id"]

        # Accounts for this customer
        acc_rows = accounts_df[accounts_df["customer_id"] == cid].to_dict(orient="records")

        if not acc_rows:
            continue  # no accounts → skip

        account_ids = {str(a["account_id"]) for a in acc_rows}

        # CHANGED: filter transactions for these accounts
        tx_rows = transactions_df[transactions_df["account_id"].astype(str).isin(account_ids)]
        tx_rows = tx_rows.to_dict(orient="records")

        if tx_rows:
            return cust, acc_rows, tx_rows

    raise RuntimeError("No customer with transactions found in dataset.")

first_customer, account_rows, txn_rows = get_case_data_with_transactions()
customer_id = first_customer["customer_id"]

logger = ExplainabilityLogger(file_path="integration_smoke_audit.jsonl", verbose=True)
loader = DataLoader(explainability_logger=logger)

case = loader.create_case_from_data(
    customer_data=first_customer,
    account_data=account_rows,
    transaction_data=txn_rows,
)

print(
    f"[DEBUG] Built CaseData: case_id={case.case_id}, "
    f"customer_id={case.customer.customer_id}, "
    f"accounts={len(case.accounts)}, txns={len(case.transactions)}"
)

# ---------- 2) Run Risk Analyst Agent (fallback mode) ----------

# Minimal mock client so RiskAnalystAgent can be constructed without real API calls.
class MockOpenAIClient:
    class Chat:
        class Completions:
            def create(self, **kwargs):
                content = json.dumps({
                    "classification": "Structuring",
                    "risk_level": "High",
                    "confidence_score": 0.9,
                    "key_indicators": ["Near-threshold deposits"],
                    "reasoning": (
                        "Step 1: Reviewed history. Step 2: Found near-threshold deposits. "
                        "Step 3: Mapped to structuring typology. "
                        "Step 4: Assessed risk as High. Step 5: Classified as Structuring."
                    )
                })
                class Message: pass
                msg = Message()
                msg.content = content

                class Choice: pass
                ch = Choice()
                ch.message = msg

                class Response: pass
                resp = Response()
                resp.choices = [ch]
                return resp
        completions = Completions()
    chat = Chat()

risk_agent = RiskAnalystAgent(
    openai_client=MockOpenAIClient(),
    explainability_logger=logger,
    model="gpt-4",
    use_mock=True  # forces use of _generate_fallback_analysis in your current code
)

risk_result = risk_agent._generate_fallback_analysis(case)

print("\n[RiskAnalystOutput]")
print("  classification      :", risk_result.classification)
print("  risk_level          :", risk_result.risk_level)
print("  confidence_score    :", risk_result.confidence_score)
print("  key_indicators      :", risk_result.key_indicators)
print("  reasoning           :", risk_result.reasoning)
if hasattr(risk_result, "confidence_rationale"):
    print("  confidence_rationale:", getattr(risk_result, "confidence_rationale"))

# ---------- 3) Run Compliance Officer Agent using risk_result + case ----------

comp_agent = ComplianceOfficerAgent(
    openai_client=MockOpenAIClient(),  # safe for now; replace with real client if desired
    explainability_logger=logger,
    use_mock=True,  # if your agent supports it; otherwise omit
)

comp_output = comp_agent._generate_fallback_narrative(case, risk_result)

print("\n[ComplianceOfficerOutput]")
print("  narrative_reasoning :", comp_output.narrative_reasoning)
print("  narrative           :", comp_output.narrative)
print("  regulatory_citations:", comp_output.regulatory_citations)
print("  completeness_check  :", comp_output.completeness_check)

print("\nIntegration smoke test complete.")

# Helper to format transactions for prompts
formatted_transactions = "\n".join(
    f"{t.transaction_id}: {t.transaction_date} {t.transaction_type} "
    f"${t.amount:,.2f} ({t.location})"
    for t in case.transactions
)

comp_agent = ComplianceOfficerAgent(
    openai_client=MockOpenAIClient(),  # safe for now; replace with real client if desired
    explainability_logger=logger,
)

# # NOTE: Adjust this method name if your ComplianceOfficerAgent uses a different fallback entry point
# comp_output = comp_agent._generate_fallback_narrative(
#     case_data=case,
#     risk_analysis=risk_result,
#     formatted_analysis=formatted_analysis,
#     formatted_transactions=formatted_transactions,
# )

print("\n[ComplianceOfficerOutput]")
print("  narrative_reasoning :", comp_output.narrative_reasoning)
print("  narrative           :", comp_output.narrative)
print("  regulatory_citations:", comp_output.regulatory_citations)
print("  completeness_check  :", comp_output.completeness_check)

print("\nIntegration smoke test complete.")

Running end-to-end integration smoke test...

Using data_dir: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter/data
DEBUG: Logging action 'create_case' for case 29729dbd-e3bc-4d82-a19f-45808432873a
[DEBUG] Built CaseData: case_id=29729dbd-e3bc-4d82-a19f-45808432873a, customer_id=CUST_0002, accounts=1, txns=11
DEBUG [RiskAnalystAgent]: Received client ID 139513897072832

[RiskAnalystOutput]
  classification      : Other
  risk_level          : Medium
  confidence_score    : 0.66
  key_indicators      : ['Total transactions: 11', 'Total amount: $29,377.86']
  reasoning           : Step 1: Data review – reviewed available customer and transaction data. Step 2: Pattern recognition – no strong match to structuring, sanctions, fraud, or money laundering typologies. Step 3: Regulatory mapping – no clear alignment with a specific suspicious activity definition, but activity warrants monitoring. Step 4: Risk quantification – residual risk present, but evidence is insufficient fo